# MongoDB Manager

Esta é uma documentação detalhada e didática da classe `MongoDBManager`, projetada para facilitar o entendimento de desenvolvedores que integrarão esta ferramenta em seus projetos.

---

## Visão Geral

A classe `MongoDBManager` atua como um **Wrapper** (ou facilitadora) para a biblioteca `pymongo`. Seu principal objetivo é simplificar a interação com bancos de dados MongoDB, abstraindo a complexidade de gerenciar conexões e garantindo que operações comuns de **CRUD** (Create, Read, Update, Delete) sejam realizadas de forma padronizada.

Ela é ideal para aplicações que precisam de persistência de dados flexível, lidando automaticamente com metadados como datas de criação e atualização.

---

## Fluxo de Execução

O funcionamento da classe segue uma lógica de **Lazy Loading** (carregamento tardio):

1. **Instanciação:** Ao criar o objeto, ele apenas armazena a URI de conexão. O banco ainda não é acessado.
2. **Primeira Chamada:** Quando qualquer método de operação (como `save_payload`) é invocado, ele chama internamente o método `connect()`.
3. **Conexão Persistente:** O `MongoClient` é criado e armazenado no atributo `self.client`. Todas as chamadas subsequentes reutilizam essa mesma conexão para economizar recursos.
4. **Processamento:** O método executa a lógica solicitada (inserção, busca, etc.).
5. **Tratamento de Erros:** Se algo falhar, a conexão é encerrada preventivamente para evitar vazamentos de memória ou conexões "presas".

---

## Tabela de Métodos

| **Método** | **Descrição Breve** |
| --- | --- |
| `__init__` | Configura a URI de conexão (via argumento ou variável de ambiente). |
| `connect` | Estabelece ou reaproveita a conexão com o servidor MongoDB. |
| `close_connection` | Finaliza a conexão ativa e limpa o cliente. |
| `save_payload` | Insere um novo documento com timestamp de criação. |
| `fetch_documents` | Busca documentos com filtros e opção de limite. |
| `update_documents` | Atualiza um ou vários registros, adicionando timestamp de edição. |
| `delete_documents` | Remove registros do banco de forma definitiva. |

---

## Arquitetura e Insights

- **Gerenciamento de Estado:** A classe utiliza o padrão *Singleton-like* para a conexão dentro da instância, garantindo que você não abra centenas de conexões desnecessárias.
- **Segurança de Dados:** No método de atualização, o código remove explicitamente a chave `_id` do payload de atualização (`new_values.pop("_id", None)`). Isso evita erros comuns do MongoDB onde se tenta sobrescrever o ID imutável de um documento.
- **Rastreabilidade:** A inclusão automática de `_created_at` e `updated_at` transforma documentos simples em registros auditáveis, facilitando a depuração e análise de dados posterior.
- **Tratamento de Erros:** O uso de `RuntimeError` encapsula erros de rede complexos em mensagens legíveis para o desenvolvedor da aplicação.

---

## Detalhamento da Classe

## Classe MongoDBManager

**Descrição**

Gerenciador central de persistência MongoDB que encapsula o ciclo de vida da conexão e operações CRUD, garantindo a reutilização de recursos e padronização de metadados.

**Argumentos**

- `mongo_uri` (str, opcional): String de conexão. Se omitida, busca em `os.getenv("MONGO_URI")` ou usa o padrão local.

---

## Métodos

## 1. connect

**Descrição:** Garante que exista uma conexão ativa com o banco.

**Argumentos:** Nenhum.

**Retornos:** `MongoClient` (objeto de conexão do pymongo).

**Raises:** `Exception` em caso de falha na rede ou credenciais.

**Exemplos:**

```bash
manager = MongoDBManager()
client = manager.connect()
```

## 2. save_payload

**Descrição:** Insere um dicionário no banco de dados e adiciona a data de criação.

**Argumentos:**

- `database_name` (str): Nome do banco.
- `collection_name` (str): Nome da coleção.
- `payload` (dict): Dados a serem salvos.
    
**Retornos:** `dict` contendo o status e o ID gerado.

**Raises:** `RuntimeError` se a inserção falhar.

**Exemplos:**

```bash
manager.save_payload("log_db", "events", {"event": "login_success"})
```

### 3. fetch_documents

**Descrição:** Localiza documentos que correspondam aos critérios informados.

**Argumentos:**

- `database_name` (str): Nome do banco.
- `collection_name` (str): Nome da coleção.
- `filter` (dict): Filtro de busca (ex: `{"status": "active"}`).
- `limit` (int): Quantidade máxima de resultados.
    
**Retornos:** `List[dict]` (Lista de documentos encontrados).

**Raises:** `RuntimeError`.

**Exemplos:**

```bash
users = manager.fetch_documents("app", "users", {"age": {"$gt": 18}}, limit=10)
```

## 4. update_documents

**Descrição:** Modifica documentos existentes e registra a data da alteração.

**Argumentos:**

- `filter` (dict): Critério para encontrar os documentos.
- `new_values` (dict): Dados a serem atualizados.
- `multi` (bool): Se `True`, atualiza todos os documentos encontrados; se `False`, apenas o primeiro.
    
**Retornos:** `dict` com métricas (`matched_count`, `modified_count`).

**Raises:** `RuntimeError`.

**Exemplos:**

```bash
manager.update_documents("app", "users", {"id": 1}, {"status": "premium"})
```

## 5. delete_documents

**Descrição:** Remove permanentemente documentos da coleção.

**Argumentos:**

- `filter` (dict): Critério de exclusão.
- `multi` (bool): Define se apaga um ou todos os registros encontrados.
    
**Retornos:** `dict` com `deleted_count`.

**Raises:** `RuntimeError`.

**Exemplos:**
    
```bash
manager.delete_documents("app", "temp_data", {"expired": True}, multi=True)
```
